# Part 10: Agentic GraphRAG**소요 시간**: 약 2시간**난이도**: ⭐⭐⭐⭐ (중상급)**목표**: LangGraph로 자율적으로 판단하고 실행하는 GraphRAG Agent를 구축한다.---## 학습 순서1. 환경 설정2. Agentic RAG 패러다임3. Graph Tool-calling 에이전트4. LangGraph 멀티에이전트5. 환각 감지 + 자가 수정6. 프로덕션 에이전트 아키텍처7. 연습 문제

---## 1. 환경 설정### 패키지 설치 및 Neo4j 연결

In [ ]:
import os, json, timefrom dotenv import load_dotenvfrom neo4j import GraphDatabaseload_dotenv()load_dotenv(dotenv_path="../.env")NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))driver.verify_connectivity()print("[OK] Neo4j 연결 성공!")def run_query(query, parameters=None, print_result=True):    with driver.session() as session:        result = session.run(query, parameters or {})        records = [record.data() for record in result]        if print_result:            for i, rec in enumerate(records, 1):                print(f"  [{i}] {rec}")        return recordsprint("[OK] 환경 설정 완료")

---## 2. Agentic RAG 패러다임### 기존 RAG vs Agentic RAG| 항목 | 기존 RAG | Agentic RAG ||------|---------|-------------|| 흐름 | 고정 파이프라인 | 동적 판단 || 검색 | 1회 검색 | 반복 검색 가능 || 오류 처리 | 없음 | 자가 수정 || 도구 | 없음 | Tool-calling |### ReAct 패턴```사용자 질문 → Thought → Action → Observation → Thought → ... → Final Answer```> **핵심**: Agent가 "어떤 도구를 쓸지" 스스로 판단하고, 결과를 보고 다시 판단

---## 3. Graph Tool-calling 에이전트### 3가지 핵심 Tool| Tool | 입력 | 출력 | 용도 ||------|------|------|------|| `schema_tool` | 없음 | 스키마 정보 | 어떤 노드/관계가 있는지 파악 || `cypher_tool` | Cypher 쿼리 | 쿼리 결과 | 정확한 그래프 탐색 || `path_tool` | 시작/끝 노드 | 경로 | Multi-hop 추론 |

In [ ]:
# Tool 정의def schema_tool():    """Neo4j 스키마 조회"""    labels = run_query("CALL db.labels() YIELD label RETURN collect(label) AS labels", print_result=False)    rels = run_query("CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS types", print_result=False)    return {        "labels": labels[0]["labels"] if labels else [],        "relationship_types": rels[0]["types"] if rels else []    }def cypher_tool(query: str):    """Cypher 쿼리 실행"""    try:        return run_query(query, print_result=False)    except Exception as e:        return {"error": str(e)}def path_tool(start: str, end: str, max_depth: int = 3):    """두 노드 사이 최단 경로"""    return run_query(f'''        MATCH path = shortestPath(            (a {{name: $start}})-[*..{max_depth}]-(b {{name: $end}})        )        RETURN [n IN nodes(path) | n.name] AS nodes,               [r IN relationships(path) | type(r)] AS relations,               length(path) AS hops    ''', parameters={"start": start, "end": end}, print_result=False)print("[OK] 3개 Tool 정의 완료")print(f"스키마: {schema_tool()}")

---## 4. LangGraph 멀티에이전트### Explorer-Reasoner-Validator 아키텍처```질문 → Explorer (그래프 탐색)          ↓       Reasoner (추론 + 답변 생성)          ↓       Validator (KG 기반 검증)          ↓       ✅ 통과 → 최종 답변       ❌ 실패 → Explorer로 재탐색 (최대 3회)```

In [ ]:
# LangGraph 멀티에이전트 (의사 코드)agent_code = '''from langgraph.graph import StateGraph, START, ENDfrom typing import TypedDict, Annotatedclass AgentState(TypedDict):    question: str    search_results: list    answer: str    is_valid: bool    retry_count: intdef explorer(state: AgentState):    # 그래프 탐색 에이전트    schema = schema_tool()    # LLM으로 Cypher 생성 → 실행    cypher = generate_cypher(state["question"], schema)    results = cypher_tool(cypher)    return {"search_results": results}def reasoner(state: AgentState):    # 추론 + 답변 생성    context = format_results(state["search_results"])    answer = llm.invoke(f"Context: {context}\nQuestion: {state['question']}")    return {"answer": answer}def validator(state: AgentState):    # KG 기반 검증    entities = extract_entities(state["answer"])    for entity in entities:        exists = cypher_tool(f"MATCH (n {{name: '{entity}'}}) RETURN n LIMIT 1")        if not exists:            return {"is_valid": False, "retry_count": state["retry_count"] + 1}    return {"is_valid": True}# StateGraph 구성workflow = StateGraph(AgentState)workflow.add_node("explorer", explorer)workflow.add_node("reasoner", reasoner)workflow.add_node("validator", validator)workflow.add_edge(START, "explorer")workflow.add_edge("explorer", "reasoner")workflow.add_edge("reasoner", "validator")# 조건부 라우팅: 검증 실패 시 재탐색workflow.add_conditional_edges("validator", lambda s:    "explorer" if not s["is_valid"] and s["retry_count"] < 3 else END)app = workflow.compile()'''print("=== LangGraph 멀티에이전트 아키텍처 ===")print(agent_code)print("\n[주의] 실행하려면 langgraph, openai 패키지가 필요합니다.")

---## 5. 환각 감지 + 자가 수정### 환각 감지 전략| 전략 | 방법 | 정확도 ||------|------|--------|| Entity 검증 | 답변의 엔티티가 KG에 존재하는지 확인 | 높음 || Relation 검증 | 답변의 관계가 KG에 존재하는지 확인 | 매우 높음 || Confidence 임계값 | LLM의 confidence < 0.7이면 재검색 | 중간 |> **핵심**: KG가 있으면 환각 감지가 쉬워진다. 벡터 RAG에서는 불가능한 기능.

In [ ]:
# 환각 감지 함수def detect_hallucination(answer: str, kg_entities: list) -> dict:    """답변에서 KG에 없는 엔티티를 찾아 환각을 감지합니다."""    # 간단한 예시: 답변에 언급된 엔티티가 KG에 있는지 확인    mentioned = []  # 실제로는 NER이나 LLM으로 추출    hallucinated = [e for e in mentioned if e not in kg_entities]    return {        "is_hallucinated": len(hallucinated) > 0,        "hallucinated_entities": hallucinated,        "confidence": 1.0 - (len(hallucinated) / max(len(mentioned), 1))    }# KG 엔티티 목록 가져오기kg_entities = run_query("MATCH (n) RETURN collect(n.name) AS names", print_result=False)print(f"KG 엔티티 수: {len(kg_entities[0]['names']) if kg_entities else 0}개")print("\n환각 감지 함수가 준비되었습니다.")

---## 6. 연습 문제### 문제 1: Tool 추가`vector_tool`을 추가하세요 — 벡터 유사도 검색 결과를 반환하는 도구.### 문제 2: 재시도 로직Validator가 실패했을 때 Explorer에 "이전 쿼리가 왜 실패했는지" 피드백을 전달하는 로직을 설계하세요.### 문제 3: Supervisor 패턴Explorer, Reasoner, Validator 위에 Supervisor 노드를 추가하여 어떤 에이전트를 호출할지 결정하는 아키텍처를 설계하세요.

In [ ]:
# [연습] 여기에 코드를 작성하세요print("Agentic GraphRAG 연습 문제를 풀어보세요!")

---## 7. 정리| 개념 | 설명 ||------|------|| Agentic RAG | 자율적 판단 + 반복 검색 || Tool-calling | schema/cypher/path 도구 || LangGraph | StateGraph 기반 워크플로우 || 환각 감지 | KG 엔티티로 검증 || 자가 수정 | 실패 → 재탐색 (최대 3회) |> **"Agent의 가치는 '실패해도 스스로 수정'하는 능력에 있다."**### 다음 Part 11: 디버깅 & 최적화

In [ ]:
# 세션 정리# driver.close()# print("[OK] Neo4j 드라이버 종료 완료")print("실습을 마칩니다. 수고하셨습니다!")